In [ ]:
### 블로그 크롤링 ###

import os
import time
import json
import urllib.parse
import urllib.request
import re
import pandas as pd
from bs4 import BeautifulSoup
import requests

# ---------------------------------------
# 1) 설정
# ---------------------------------------
CLIENT_ID = os.getenv("NAVER_CLIENT_ID", "여기 수정") ##네이버 클라이언트 ID "여기수정"에 넣기
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET", "여기 수정") ##네이버 클라이언트 시크릿 "여기 수정"에 넣기

KEYWORD = "GNL"     
END_PAGES = 50             
DISPLAY = 20              
OUTPUT_CSV = "blog.csv"   ##출력되는 파일 이름

# ✅ 검색 기간 설정 (YYYYMMDD 형식)
FROM_DATE = "20231111"   # 2022-11-11 <--- 날짜 형식임
TO_DATE   = "20251111"   # 2025-11-11

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"
}

# ---------------------------------------
# 2) 유틸 함수
# ---------------------------------------
def validate_keys():
    if not CLIENT_ID or not CLIENT_SECRET:
        raise SystemExit("[오류] 네이버 API Client ID/Secret 확인 필요.")

def clamp_display(n:int) -> int:
    return min(100, max(10, n))

def naver_blog_search(query: str, start: int = 1, display: int = 10):
    display = clamp_display(display)
    base = "https://openapi.naver.com/v1/search/blog.json"
    params = {"query": query, "start": start, "display": display, "sort": "sim"}
    url = f"{base}?{urllib.parse.urlencode(params)}"

    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", CLIENT_ID)
    req.add_header("X-Naver-Client-Secret", CLIENT_SECRET)

    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode("utf-8")).get("items", [])

def to_mobile_url(url: str) -> str:
    return url.replace("://blog.naver.com", "://m.blog.naver.com")

def clean_html_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", "", text)
    return re.sub(r"\s+", " ", text.replace("\u200b","").replace("\xa0"," ")).strip()

def extract_blog_content(url: str):
    r = requests.get(to_mobile_url(url), headers=HEADERS, timeout=12, allow_redirects=True)
    soup = BeautifulSoup(r.text, "lxml")

    og = soup.find("meta", property="og:title")
    title = clean_html_text(og["content"]) if og and og.get("content") else ""

    date = ""
    for sel in ["span.blog_date","time.se_publishDate","span.se_publishDate"]:
        d = soup.select_one(sel)
        if d:
            date = clean_html_text(d.get_text())
            break

    body = soup.select_one("div.se-main-container") or soup.select_one("div#postViewArea")
    content = clean_html_text(str(body)) if body else ""

    return title, content, date

# ---------------------------------------
# 3) 메인 실행 + ✅ 기간 필터링 추가
# ---------------------------------------
def run(keyword: str, end_pages: int, display: int, outfile: str):
    validate_keys()
    display = clamp_display(display)

    all_rows = []

    for page in range(end_pages):
        start = page * display + 1
        items = naver_blog_search(keyword, start=start, display=display)

        if not items:
            break

        for it in items:
            link = it.get("link", "")
            postdate = it.get("postdate", "")  # YYYYMMDD

            # ✅ 날짜 필터링 수행
            if not (FROM_DATE <= postdate <= TO_DATE):
                continue

            if "blog.naver.com" not in link:
                continue

            try:
                title, content, date = extract_blog_content(link)
                all_rows.append({
                    "title": title or clean_html_text(it.get("title","")),
                    "content": content,
                    "date": date or postdate,
                    "url": link
                })
                time.sleep(0.6)

            except Exception as e:
                print("[경고] 본문 파싱 실패:", link, e)

    if not all_rows:
        print("[안내] 조건에 맞는 데이터가 없습니다.")
        return

    df = pd.DataFrame(all_rows).drop_duplicates(subset=["url"])
    df.to_csv(outfile, index=False, encoding="utf-8-sig")
    print(f"[완료] {len(df)}건 저장 → {outfile}")

if __name__ == "__main__":
    run(KEYWORD, END_PAGES, DISPLAY, OUTPUT_CSV)
